# 🌌 Módulo 05: Modelos Generativos y Espacios Latentes
## Capítulo 1: Autoencoders, Inferencia Variacional (VAEs) y el Reparameterization Trick

> *"Un autoencoder clásico comprime datos en un espacio latente discreto lleno de agujeros vacíos: si eliges un punto al azar en medio, el decodificador genera ruido incomprensible. Diederik Kingma y Max Welling se hicieron una pregunta fundamental: ¿cómo forzar al espacio latente a ser continuo, denso y muestreable? Su respuesta fue el Variational Autoencoder (VAE) y uno de los trucos analíticos más hermosos de la IA: el Reparameterization Trick."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/05_generative_models/01_autoencoders_and_vaes.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos librerías matemáticas y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar Autoencoders y VAEs from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Nacimiento de los Autoencoders (1986 - 2006)
En 1986, Rumelhart, Hinton y Williams propusieron entrenar una red con forma de reloj de arena (*hourglass*) para reconstruir su propia entrada: $x \to z \to \hat{x}$.
En 2006, **Geoffrey Hinton y Ruslan Salakhutdinov** publicaron en *Science* su célebre artículo sobre reducción de dimensionalidad no lineal mediante autoencoders profundos.

### La Gran Limitación: El Espacio Latente Discontinuo
Un Autoencoder clásico es excelente para comprimir o eliminar ruido, pero **no es un modelo generativo**:
* Mapea cada imagen $x$ a un único punto determinista $z \in \mathbb{R}^d$.
* El modelo aprende a colocar cada clase en "islas aisladas" separadas por enormes abismos vacíos para minimizar el error de reconstrucción.
* Si tomas un punto aleatorio en el espacio latente que cae en uno de esos abismos, **el decodificador genera un patrón borroso o carente de sentido físico**.

### Diederik Kingma & Max Welling (2013): Variational Autoencoders (VAEs)
A finales de 2013, Kingma y Welling (y paralelamente Danilo Rezende en DeepMind) revolucionaron el campo con el **VAE**:
* En lugar de mapear la entrada $x$ a un vector determinista $z$, el codificador predice los parámetros de una distribución de probabilidad: una **media** $\mu(x)$ y una **varianza** $\sigma^2(x)$.
* Forzaron a que la distribución latente se mantenga cercana a una distribución normal estándar $\mathcal{N}(0, I)$ mediante la **Divergencia KL**.

### El Muro Matemático y el Momento Eureka: El Reparameterization Trick
Para pasar el vector latente al decodificador, necesitamos muestrear $z \sim \mathcal{N}(\mu, \sigma^2)$.
¡Pero una operación de muestreo aleatorio estocástico rompe el grafo computacional! No se puede calcular la derivada $\frac{\partial z}{\partial \mu}$ ni $\frac{\partial z}{\partial \sigma}$ a través de un nodo aleatorio.

**La genial solución analítica (El truco de reparametrización):**
Desacoplamos la fuente de aleatoriedad del grafo de parámetros:
$$z = \mu + \sigma \odot \epsilon, \quad \text{donde } \epsilon \sim \mathcal{N}(0, I)$$

El ruido $\epsilon$ entra como una señal externa fija sin gradiente. Las operaciones de suma y multiplicación por $\mu$ y $\sigma$ son **100% analíticamente diferenciables**: ¡Backpropagation vuelve a funcionar a la perfección!

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### La Función de Pérdida ELBO (Evidence Lower Bound)
La pérdida total de un VAE es una balanza entre dos fuerzas opuestas:
$$\mathbf{\mathcal{L}_{VAE} = \text{Pérdida de Reconstrucción} + D_{KL}(q(z|x) \parallel \mathcal{N}(0, I))}$$

1. **Término de Reconstrucción (Fidelidad):**
   $$\text{Recon Loss} = \|x - \hat{x}\|^2 \quad (\text{o Binary Cross-Entropy})$$
   * *Fuerza centrífuga:* Empuja al codificador a separar las clases en distintas regiones para no confundir un gato con un perro.

2. **Divergencia de Kullback-Leibler (Regularización Bayesiana):**
   Mide cuánto se desvía la distribución estimada $\mathcal{N}(\mu, \sigma^2)$ de la normal estándar $\mathcal{N}(0, I)$:
   $$D_{KL} = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$
   * *Fuerza centrípeta:* Empuja a todas las distribuciones hacia el origen $\mu = 0$ con dispersión $\sigma = 1$.

### El Equilibrio Dinámico:
* Si solo existiera la Reconstrucción: El espacio latente se fragmentaría en islas aisladas (Autoencoder clásico).
* Si solo existiera la Divergencia KL: Todo colapsaría a una nube gaussiana uniforme sin recordar los datos.
* **El equilibrio entre ambas genera un espacio latente continuo, suave y sin huecos**, donde caminar en línea recta entre dos puntos produce una metamorfosis visual coherente.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Construyamos un **Autoencoder Clásico** y un **Variational Autoencoder (VAE)** con el *Reparameterization Trick*.

In [ ]:
class StandardAutoencoder(nn.Module):
    """
    Autoencoder clásico determinista: x -> z (fijo) -> x_hat
    """
    def __init__(self, in_dim: int = 2, latent_dim: int = 2, hidden_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim)
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z


class VariationalAutoencoder(nn.Module):
    """
    VAE completo con cabezales de mu y log(sigma^2) y Reparameterization Trick.
    """
    def __init__(self, in_dim: int = 2, latent_dim: int = 2, hidden_dim: int = 32):
        super().__init__()
        # Backbone compartido del codificador
        self.enc_shared = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU()
        )
        # Cabezales para los parámetros gaussianos
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)  # log(sigma^2) para estabilidad numérica
        
        # Decodificador generativo p(x|z)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim)
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.enc_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        El Reparameterization Trick: z = mu + sigma * epsilon
        donde epsilon ~ N(0, I) es ruido estocástico exógeno.
        """
        if self.training:
            std = torch.exp(0.5 * logvar)  # sigma = exp(0.5 * log(sigma^2))
            eps = torch.randn_like(std)     # Muestreo de ruido blanco
            return mu + std * eps
        else:
            # En inferencia determinista usamos directamente la media
            return mu

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

print("✅ StandardAutoencoder y VariationalAutoencoder compilados exitosamente")

### Función de Pérdida ELBO y Entrenamiento Comparativo
Generamos un conjunto de datos sintético bidimensional compuesto por 4 cúmulos gaussianos en las esquinas para comparar la geometría latente aprendida por el Autoencoder Clásico frente al VAE:

In [ ]:
# Pérdida ELBO del VAE
def vae_loss_function(recon_x, x, mu, logvar, beta=1.0):
    # 1. Error de reconstrucción cuadrático (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    # 2. Divergencia KL analítica: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_div

# Generar datos de prueba: 4 centros en 2D
np.random.seed(42)
centers = np.array([[-2, -2], [-2, 2], [2, -2], [2, 2]])
X_data = []
labels = []
for idx, c in enumerate(centers):
    X_data.append(np.random.randn(250, 2) * 0.4 + c)
    labels.append(np.full(250, idx))
X_data = np.vstack(X_data).astype(np.float32)
labels = np.concatenate(labels)
t_X = torch.from_numpy(X_data)

# Entrenar Autoencoder Clásico
ae = StandardAutoencoder(in_dim=2, latent_dim=2)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-2)
for _ in range(250):
    opt_ae.zero_grad()
    recon, _ = ae(t_X)
    loss = F.mse_loss(recon, t_X, reduction='sum')
    loss.backward()
    opt_ae.step()

# Entrenar Variational Autoencoder (VAE)
vae = VariationalAutoencoder(in_dim=2, latent_dim=2)
opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-2)
for _ in range(250):
    opt_vae.zero_grad()
    recon, mu, logvar = vae(t_X)
    loss = vae_loss_function(recon, t_X, mu, logvar, beta=0.5)
    loss.backward()
    opt_vae.step()

print("✅ Ambos modelos entrenados exitosamente sobre el dataset sintético")

### Experimento Visual: Comparación Geométrica del Espacio Latente
Proyectemos los datos en el espacio latente 2D de ambos modelos para observar el efecto de la regularización KL:

In [ ]:
ae.eval()
vae.eval()
with torch.no_grad():
    _, z_ae = ae(t_X)
    mu_vae, _ = vae.encode(t_X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Espacio Latente del Autoencoder Clásico
scatter1 = axes[0].scatter(z_ae[:, 0], z_ae[:, 1], c=labels, cmap='tab10', alpha=0.7, edgecolors='none')
axes[0].set_title("Autoencoder Clásico: Islas Aisladas y Discontinuas\n(No se puede muestrear uniformemente)", fontsize=11)
axes[0].set_xlabel("Latente z_1")
axes[0].set_ylabel("Latente z_2")
axes[0].grid(True, linestyle=':', alpha=0.5)

# 2. Espacio Latente del VAE
scatter2 = axes[1].scatter(mu_vae[:, 0], mu_vae[:, 1], c=labels, cmap='tab10', alpha=0.7, edgecolors='none')
axes[1].set_title("VAE: Espacio Latente Regularizado Centrado en N(0, I)\n(Espacio suave y muestreable)", fontsize=11)
axes[1].set_xlabel("Latente z_1")
axes[1].set_ylabel("Latente z_2")
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()
print("👁️ Observa cómo el VAE atrae todos los cúmulos hacia el centro N(0, I) sin dejar huecos muertos.")

---

## 4. ⚡ Transición a PyTorch Moderno

Verifiquemos la fórmula analítica de la divergencia KL entre dos distribuciones normales multivariantes:

In [ ]:
# Si mu = 0 y logvar = 0 (sigma^2 = 1, es decir, N(0, I)), la divergencia KL debe ser exactamente 0.0
mu_zero = torch.zeros(1, 4)
logvar_zero = torch.zeros(1, 4)
kl_ideal = -0.5 * torch.sum(1 + logvar_zero - mu_zero.pow(2) - logvar_zero.exp())
assert torch.isclose(kl_ideal, torch.tensor(0.0))
print(f"KL para N(0, I) vs N(0, I): {kl_ideal.item():.4f} (Coincidencia exacta)")

# Si mu = [1, -1] y sigma^2 = [2, 2]
mu_test = torch.tensor([[1.0, -1.0]])
logvar_test = torch.tensor([[np.log(2.0), np.log(2.0)]])
# Formula: -0.5 * (1 + log(2) - 1 - 2) * 2 = -(log(2) - 2) = 2 - log(2) = 1.30685
kl_manual = -0.5 * torch.sum(1 + logvar_test - mu_test.pow(2) - logvar_test.exp())
expected_kl = float(2.0 * (-0.5 * (1.0 + np.log(2.0) - 1.0 - 2.0)))
assert np.isclose(kl_manual.item(), expected_kl, atol=1e-4)
print(f"KL calculada: {kl_manual.item():.4f} == Esperada: {expected_kl:.4f}")
print("✅ La formulación analítica de la divergencia KL es numéricamente exacta")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Generación de Datos Muestreando de la Prior $\mathcal{N}(0, I)$
Como el VAE garantiza que el espacio latente se distribuye según $\mathcal{N}(0, I)$, podemos **generar nuevas muestras sin ver ninguna imagen de entrada**, muestreando puntos aleatorios del prior y pasándolos por el decodificador:

In [ ]:
# Generar 200 puntos aleatorios puros desde el prior N(0, I)
z_samples = torch.randn(200, 2)
vae.eval()
with torch.no_grad():
    x_generated = vae.decoder(z_samples).numpy()

plt.figure(figsize=(6, 5))
plt.scatter(X_data[:, 0], X_data[:, 1], c='gray', alpha=0.2, label='Datos Originales Reales')
plt.scatter(x_generated[:, 0], x_generated[:, 1], c='crimson', marker='x', alpha=0.7, label='Muestras Sintéticas Generadas')
plt.title('Síntesis Generativa Muestreando desde el Prior N(0, I)', fontsize=11, fontweight='bold')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()
print("🚀 ¡El VAE genera puntos válidos que caen de forma natural sobre los cúmulos de datos aprendidos!")

### Reto 2 (Para resolver): Implementar Interpolación Lineal Esférica (Slerp)
En espacios latentes de alta dimensión ($d > 10$), una línea recta euclidiana ($z(t) = (1-t) z_1 + t z_2$) pasa por el centro del espacio, donde la densidad de probabilidad gaussiana decae. La interpolación **Slerp (Spherical Linear Interpolation)** recorre la superficie de la hiperesfera preservando la norma constante:
$$\text{Slerp}(z_1, z_2, t) = \frac{\sin((1-t)\Omega)}{\sin(\Omega)} z_1 + \frac{\sin(t\Omega)}{\sin(\Omega)} z_2, \quad \text{donde } \cos(\Omega) = \frac{z_1 \cdot z_2}{\|z_1\| \|z_2\|}$$

Implementa a continuación la función `slerp(z1, z2, t)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def slerp(z1: torch.Tensor, z2: torch.Tensor, t: float) -> torch.Tensor:
    """
    Calcula la interpolación esférica (Slerp) entre los vectores latentes z1 y z2 para un t in [0, 1].
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Kingma, D. P., & Welling, M. (2013):** *"Auto-Encoding Variational Bayes"*, ICLR 2014. [arXiv:1312.6114](https://arxiv.org/abs/1312.6114)
   * *¿Qué leer?* Sección 2 ("Method") y Sección 2.4 ("The Reparameterization Trick"): la formalización del VAE.
2. **Rezende, D. J., Mohamed, S., & Wierstra, D. (2014):** *"Stochastic Backpropagation and Approximate Inference in Deep Generative Models"*, ICML 2014. [arXiv:1401.4082](https://arxiv.org/abs/1401.4082)
   * *¿Qué leer?* El enfoque de DeepMind derivando el mismo principio variacional.
3. **Higgins, I., et al. (2017):** *"beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework"*, ICLR 2017. [OpenReview](https://openreview.net/forum?id=Sy2fzU9gl)
   * *¿Qué leer?* El factor de ponderación $\beta$ en la divergencia KL para lograr desenredo latente (*latent disentanglement*).
4. **Doersch, C. (2016):** *"Tutorial on Variational Autoencoders"*, arXiv:1606.05908. [arXiv Link](https://arxiv.org/abs/1606.05908)
   * *¿Qué leer?* La mejor guía pedagógica y matemática paso a paso para entender la inferencia variacional.